In [27]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline


In [2]:
# -----------------------------------
# 1. 데이터 불러오기
# -----------------------------------
df = pd.read_csv("data/ecommerce_customer_data.csv")

# 데이터 확인
print("데이터 크기:", df.shape)
print(df.head())

데이터 크기: (2000, 9)
   Customer_ID  Age  Gender  Annual_Income  Spending_Score  Membership_Years  \
0            1   56    Male          69812              88               3.2   
1            2   69  Female          70500              26               4.3   
2            3   46  Female          99151              17               8.2   
3            4   32    Male          78643              71               0.6   
4            5   60  Female          64900              13               6.7   

   Online_Purchases  Discount_Usage  Churn  
0                92            0.43      1  
1                30            0.23      1  
2               199            0.52      0  
3               153            0.25      0  
4               127            0.94      0  


In [3]:
# -----------------------------------
# 2. 입력(X), 타깃(y) 분리
# -----------------------------------
X = df.drop(columns=["Customer_ID", "Churn"])
y = df["Churn"]

# -----------------------------------
# 3. 범주형 변수 인코딩
# Gender를 숫자로 변환
# -----------------------------------
X = pd.get_dummies(X, columns=["Gender"], drop_first=True)

print("\n전처리 후 입력 데이터 컬럼:")
print(X.columns)

# -----------------------------------
# 4. train / test 분리
# stratify=y 로 클래스 비율 유지
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n훈련 데이터 크기:", X_train.shape)
print("테스트 데이터 크기:", X_test.shape)

# -----------------------------------
# 5. Logistic Regression용 스케일링
# 트리 계열은 스케일링 없이도 됨
# -----------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


전처리 후 입력 데이터 컬럼:
Index(['Age', 'Annual_Income', 'Spending_Score', 'Membership_Years',
       'Online_Purchases', 'Discount_Usage', 'Gender_Male'],
      dtype='str')

훈련 데이터 크기: (1600, 7)
테스트 데이터 크기: (400, 7)


In [53]:
# -----------------------------------
# 6. 모델 생성
# -----------------------------------
log_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

tree_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    criterion="gini",
    random_state=42,
    class_weight="balanced"
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

In [56]:
# -----------------------------------
# 7. Logistic Regression 학습 및 평가
# -----------------------------------
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

print("\n" + "="*50)
print("Logistic Regression 결과")
print("="*50)
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("Classification Report:\n", classification_report(y_test, y_pred_log))


Logistic Regression 결과
Accuracy: 0.4825
Confusion Matrix:
 [[134 142]
 [ 65  59]]
Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.49      0.56       276
           1       0.29      0.48      0.36       124

    accuracy                           0.48       400
   macro avg       0.48      0.48      0.46       400
weighted avg       0.56      0.48      0.50       400



In [57]:
# -----------------------------------
# 8. Decision Tree 학습 및 평가
# -----------------------------------
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

print("\n" + "="*50)
print("Decision Tree 결과")
print("="*50)
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tree))
print("Classification Report:\n", classification_report(y_test, y_pred_tree))


Decision Tree 결과
Accuracy: 0.45
Confusion Matrix:
 [[115 161]
 [ 59  65]]
Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.42      0.51       276
           1       0.29      0.52      0.37       124

    accuracy                           0.45       400
   macro avg       0.47      0.47      0.44       400
weighted avg       0.55      0.45      0.47       400



In [58]:
# -----------------------------------
# 9. Random Forest 학습 및 평가
# -----------------------------------
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\n" + "="*50)
print("Random Forest 결과")
print("="*50)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest 결과
Accuracy: 0.5675
Confusion Matrix:
 [[205  71]
 [102  22]]
Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.74      0.70       276
           1       0.24      0.18      0.20       124

    accuracy                           0.57       400
   macro avg       0.45      0.46      0.45       400
weighted avg       0.53      0.57      0.55       400



In [59]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ]
})

print("\n모델별 정확도 비교")
print(results)


모델별 정확도 비교
                 Model  Accuracy
0  Logistic Regression    0.4825
1        Decision Tree    0.4500
2        Random Forest    0.5675


In [65]:
voting_model = VotingClassifier(
    estimators=[
        ("lr", log_model),
        ("dt", tree_model),
        ("rf", rf_model)
    ],
    voting="soft"
)

voting_model.fit(X_train, y_train)
y_prob_vote = voting_model.predict_proba(X_test)[:, 1]
y_pred_vote = (y_prob_vote >= 0.7).astype(int)

In [66]:
print("\n" + "="*50)
print("Voting model 결과")
print("="*50)
print("Accuracy:", accuracy_score(y_test, y_pred_vote))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_vote))
print("Classification Report:\n", classification_report(y_test, y_pred_vote))


Voting model 결과
Accuracy: 0.69
Confusion Matrix:
 [[276   0]
 [124   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       276
           1       0.00      0.00      0.00       124

    accuracy                           0.69       400
   macro avg       0.34      0.50      0.41       400
weighted avg       0.48      0.69      0.56       400



c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [68]:
# 저장
with open("voting_model.pkl", "wb") as f:
    pickle.dump(voting_model, f)

In [69]:
# 불러오기
with open("voting_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)